# Import Required Libraries

In [15]:
import pandas as pd
import numpy as np
import re
import pickle

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Load Dataset

In [16]:
df = pd.read_csv("fake_genuine_large_high_confidence_reviews.csv")
df.head()

,review_text,label
0,"Value for money, build quality is 8.",1
1,"Satisfied with the purchase, would 3 recommend.",1
2,"Value for money, build quality is 2.",1
3,The product performs 4 and matches the descrip...,1
4,I have been using this product for 10 months a...,1


# Basic Dataset Check

In [17]:
print(df.shape)
print(df['label'].value_counts())

(12000, 2)
label
1    6000
0    6000
Name: count, dtype: int64


# Text Cleaning Function

In [18]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['review_text'] = df['review_text'].apply(clean_text)

# Define Features & Target

In [19]:
X = df['review_text']
y = df['label']

# Train-Test Split

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# TF-IDF Vectorization (Accuracy Booster)

In [21]:
tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1,2),
    stop_words='english',
    min_df=2
)

X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

# Train Logistic Regression Model

In [22]:
model = LogisticRegression(
    max_iter=200,
    class_weight='balanced'
)

model.fit(X_train_vec, y_train)

LogisticRegression(class_weight='balanced', max_iter=200)

# Model Evaluation (CHECK ACCURACY)

In [23]:
y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_pred, y_pred))

Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1200
           1       1.00      1.00      1.00      1200

    accuracy                           1.00      2400
   macro avg       1.00      1.00      1.00      2400
weighted avg       1.00      1.00      1.00      2400



# Save Model & vectorizer

In [24]:
import os

# Create model folder if not exists
os.makedirs("model", exist_ok=True)

pickle.dump(model, open("model/fake_review_model.pkl", "wb"))
pickle.dump(tfidf, open("model/tfidf_vectorizer.pkl", "wb"))

print("✅ Model and Vectorizer saved successfully!")

✅ Model and Vectorizer saved successfully!
